The objective is to establish benchmark performance using all statistically selected features. Later, we'll compare these results against models trained after advanced feature selection.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
# Models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier
# Metrics
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

In [2]:
df = pd.read_csv('D:/Study/IFRS-9-Complient-Risk-Analysis-modelling/data/encoded_dataset.csv')
print(df.shape)

(51215, 57)


In [3]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

df["Approved_Flag"] = label_encoder.fit_transform(df["Approved_Flag"])

In [4]:
# Target variable
y = df["Approved_Flag"]

# Features
X = df.drop(columns=["Approved_Flag"])

**Split the Training and Testing Data with equal representation**

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape)
print(X_test.shape)

(40972, 56)
(10243, 56)


### Verify The Class Distribution

In [6]:
print("Train Distribution")
print(y_train.value_counts(normalize=True))

print("\nTest Distribution")
print(y_test.value_counts(normalize=True))

Train Distribution
Approved_Flag
1    0.627331
2    0.145270
3    0.114590
0    0.112809
Name: proportion, dtype: float64

Test Distribution
Approved_Flag
1    0.627355
2    0.145270
3    0.114615
0    0.112760
Name: proportion, dtype: float64


- The percentage of the distribution is almost identical in both the Training and Testing Data

### Creating an Evaluation Function

In [7]:
def evaluate_model(model, X_train, X_test, y_train, y_test, scale=False):

    # Create pipeline
    if scale:
        pipeline = Pipeline([
            ("scaler", StandardScaler()),
            ("model", model)
        ])
    else:
        pipeline = Pipeline([
            ("model", model)
        ])

    # Train
    pipeline.fit(X_train, y_train)

    # Predict
    y_pred = pipeline.predict(X_test)

    # Metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average="weighted")
    recall = recall_score(y_test, y_pred, average="weighted")
    f1 = f1_score(y_test, y_pred, average="weighted")

    print("=" * 60)
    print(type(model).__name__)
    print("=" * 60)

    print(f"Accuracy : {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall   : {recall:.4f}")
    print(f"F1 Score : {f1:.4f}")

    print("\nClassification Report\n")
    print(classification_report(y_test, y_pred))

    print("\nConfusion Matrix\n")
    print(confusion_matrix(y_test, y_pred))

    return pipeline, {
    "Model": type(model).__name__,
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "F1": f1
}

### ***1. Logistic Regression***

In [8]:
lr = LogisticRegression(
    max_iter=10000,
    random_state=42
)

lr_pipeline , lr_result = evaluate_model(
    lr,
    X_train,
    X_test,
    y_train,
    y_test,
    scale=True
)


LogisticRegression
Accuracy : 0.7626
Precision: 0.7232
Recall   : 0.7626
F1 Score : 0.7268

Classification Report

              precision    recall  f1-score   support

           0       0.80      0.70      0.75      1155
           1       0.79      0.94      0.86      6426
           2       0.41      0.12      0.19      1488
           3       0.70      0.64      0.67      1174

    accuracy                           0.76     10243
   macro avg       0.67      0.60      0.62     10243
weighted avg       0.72      0.76      0.73     10243


Confusion Matrix

[[ 809  342    2    2]
 [ 164 6065  124   73]
 [  39 1013  181  255]
 [   0  281  137  756]]


The large gap between Weighted F1 (0.69) and Macro F1 (0.56) immediately tells us something important:

The model performs much better on the majority class (P2) than on the minority classes (P1, P3, P4).

This is expected because your dataset is highly imbalanced.

### DecisionTree Classifier

In [10]:
dt = DecisionTreeClassifier(
    random_state=42
)

dt_pipeline , dt_result = evaluate_model(
    dt,
    X_train,
    X_test,
    y_train,
    y_test,
    scale=False
)

DecisionTreeClassifier
Accuracy : 0.7165
Precision: 0.7200
Recall   : 0.7165
F1 Score : 0.7182

Classification Report

              precision    recall  f1-score   support

           0       0.71      0.71      0.71      1155
           1       0.83      0.82      0.83      6426
           2       0.31      0.33      0.32      1488
           3       0.64      0.62      0.63      1174

    accuracy                           0.72     10243
   macro avg       0.62      0.62      0.62     10243
weighted avg       0.72      0.72      0.72     10243


Confusion Matrix

[[ 823  276   55    1]
 [ 293 5291  708  134]
 [  43  672  492  281]
 [   1  129  311  733]]


### RandomForest Classifier

In [11]:
rf = RandomForestClassifier(
    random_state=42
)

rf_pipeline , rf_result = evaluate_model(
    rf,
    X_train,
    X_test,
    y_train,
    y_test
)

RandomForestClassifier
Accuracy : 0.7823
Precision: 0.7549
Recall   : 0.7823
F1 Score : 0.7603

Classification Report

              precision    recall  f1-score   support

           0       0.83      0.73      0.78      1155
           1       0.82      0.94      0.87      6426
           2       0.44      0.22      0.29      1488
           3       0.74      0.70      0.72      1174

    accuracy                           0.78     10243
   macro avg       0.71      0.65      0.67     10243
weighted avg       0.75      0.78      0.76     10243


Confusion Matrix

[[ 846  309    0    0]
 [ 131 6020  233   42]
 [  37  884  329  238]
 [   0  168  188  818]]


### XGBoost Classifier

In [12]:
xgb_model = XGBClassifier(
    objective="multi:softprob",
    num_class=4,
    random_state=42,
    eval_metric="mlogloss"
)

xgb_pipeline , xgb_result = evaluate_model(
    xgb_model,
    X_train,
    X_test,
    y_train,
    y_test,
    scale=False
)

XGBClassifier
Accuracy : 0.7887
Precision: 0.7695
Recall   : 0.7887
F1 Score : 0.7761

Classification Report

              precision    recall  f1-score   support

           0       0.80      0.78      0.79      1155
           1       0.84      0.92      0.88      6426
           2       0.44      0.30      0.36      1488
           3       0.75      0.72      0.73      1174

    accuracy                           0.79     10243
   macro avg       0.71      0.68      0.69     10243
weighted avg       0.77      0.79      0.78     10243


Confusion Matrix

[[ 904  248    3    0]
 [ 181 5886  320   39]
 [  44  756  448  240]
 [   0   96  237  841]]


### LightGBM

In [13]:
from lightgbm import LGBMClassifier

lgbm = LGBMClassifier(
    random_state=42,
    n_estimators=200
)

lgbm_pipeline , lgbm_result = evaluate_model(
    lgbm,
    X_train,
    X_test,
    y_train,
    y_test,
    scale=False
)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.003022 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4079
[LightGBM] [Info] Number of data points in the train set: 40972, number of used features: 56
[LightGBM] [Info] Start training from score -2.182061
[LightGBM] [Info] Start training from score -0.466281
[LightGBM] [Info] Start training from score -1.929162
[LightGBM] [Info] Start training from score -2.166391
LGBMClassifier
Accuracy : 0.7918
Precision: 0.7715
Recall   : 0.7918
F1 Score : 0.7782

Classification Report

              precision    recall  f1-score   support

           0       0.79      0.79      0.79      1155
           1       0.84      0.92      0.88      6426
           2       0.45      0.30      0.36      1488
           3       0.75      0.72      0.74      1174

    accuracy                           0.79     

In [14]:
results = pd.DataFrame([
    lr_result,
    dt_result,
    rf_result,
    xgb_result,
    lgbm_result
])

results.sort_values(by="F1", ascending=False)

,Model,Accuracy,Precision,Recall,F1
4,LGBMClassifier,0.791760,0.771505,0.791760,0.778207
3,XGBClassifier,0.788734,0.769489,0.788734,0.776089
2,RandomForestClassifier,0.782290,0.754866,0.782290,0.760271
0,LogisticRegression,0.762570,0.723229,0.762570,0.726764
1,DecisionTreeClassifier,0.716489,0.720012,0.716489,0.718201


In [16]:
from pathlib import Path
import joblib
PROJECT_ROOT = Path.cwd().parent      # if notebook is inside notebooks/
MODEL_DIR = Path("../models")
DATA_DIR = Path("../data")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

### ***Save the Training and Testing Data***

In [18]:
X_train.to_csv(DATA_DIR / "processed/base_X_train.csv")
y_train.to_csv(DATA_DIR / "processed/base_y_train.csv")
X_test.to_csv(DATA_DIR / "processed/base_X_test.csv")
y_test.to_csv(DATA_DIR / "processed/base_y_test.csv")

### ***Save the baseline pipelines***

In [ ]:

joblib.dump(
    lr_pipeline,
    MODEL_DIR / "baseline_lr_pipe.pkl"
)
joblib.dump(
    dt_pipeline,
    MODEL_DIR / "baseline_dt_pipe.pkl"
)
joblib.dump(
    rf_pipeline,
    MODEL_DIR / "baseline_rf_pipe.pkl"
)
joblib.dump(
    xgb_pipeline,
    MODEL_DIR / "baseline_xgb_pipe.pkl"
)
joblib.dump(
    lgbm_pipeline,
    MODEL_DIR / "baseline_lgbm_pipe.pkl"
)

['..\\models\\baseline_lgbm_pipe.pkl']

## 4. Baseline Model Comparison

To establish a performance benchmark before feature selection and hyperparameter tuning, five machine learning models were trained and evaluated on the preprocessed dataset.

### Model Performance Summary

| Rank | Model | Accuracy | Precision | Recall | F1 Score |
|------|--------------------------|---------:|----------:|-------:|---------:|
| 🥇 1 | LightGBM | **79.18%** | **77.15%** | **79.18%** | **77.82%** |
| 🥈 2 | XGBoost | 78.87% | 76.95% | 78.87% | 77.61% |
| 🥉 3 | Random Forest | 78.23% | 75.49% | 78.23% | 76.03% |
| 4 | Logistic Regression | 76.26% | 72.32% | 76.26% | 72.68% |
| 5 | Decision Tree | 71.65% | 72.00% | 71.65% | 71.82% |

---

## Key Observations

### 1. LightGBM achieved the best overall performance.
LightGBM obtained the highest Accuracy (79.18%) and Weighted F1-score (77.82%), indicating its superior ability to model the complex, non-linear relationships present in the credit risk dataset.

### 2. XGBoost performed almost identically to LightGBM.
XGBoost achieved performance very close to LightGBM, with only a marginal difference in Accuracy and F1-score. This suggests that gradient boosting methods are particularly effective for this classification problem.

### 3. Random Forest provided a strong ensemble baseline.
Random Forest outperformed both Logistic Regression and Decision Tree, demonstrating the effectiveness of ensemble learning over a single decision tree.

### 4. Scaling significantly improved Logistic Regression.
After applying feature scaling through a preprocessing pipeline, Logistic Regression showed a noticeable improvement over the initial baseline and produced competitive results among the traditional machine learning models.

### 5. Decision Tree produced the weakest results.
Although Decision Tree successfully captured non-linear relationships, it suffered from lower generalization performance compared to ensemble methods, making it the weakest performer among the evaluated models.

---

## Business Interpretation

The results indicate that ensemble-based gradient boosting algorithms are the most suitable candidates for the IFRS 9 credit risk classification task. Their superior predictive performance suggests a better ability to distinguish between different consumer risk categories compared to linear and single-tree models.

Although LightGBM currently provides the best baseline performance, the performance gap between LightGBM and XGBoost is relatively small. Therefore, both models will be considered for further optimization through feature selection and hyperparameter tuning before selecting the final production model.

---

## Conclusion

The baseline evaluation establishes a strong benchmark for subsequent model improvements. Based on the current results:

- **Primary Candidate:** LightGBM
- **Secondary Candidate:** XGBoost

The next phase of the project will focus on **feature importance analysis**, **advanced feature selection**, and **hyperparameter tuning** to further improve model performance while maintaining interpretability and robustness for IFRS 9 compliant credit risk assessment.